In [1]:

# Analysis Plan
print("="*80)
print("ANALYSIS PLAN FOR PERTURBED DAVENPORT-HEILBRONN FUNCTION")
print("="*80)
print()
print("Objective: Test if R_comp is a general detector for non-multiplicative resonance")
print("by analyzing the perturbed function L_DH^(ε=0.01)")
print()
print("Steps:")
print("1. Implement the perturbed Davenport-Heilbronn function L_DH^(ε=0.01)")
print(" - Use κ' = κ + ε where κ = (√5-1) / (2√(5(√5-1)))")
print(" - Coefficients: a_n = ((1-iκ')/2)χ(n) + ((1+iκ')/2)χ̄(n)")
print(" - Set a_n = 0 for non-squarefree n (critical constraint)")
print()
print("2. Compute partial sum D(t;N) = Σ_{n≤N} a_n / n^(1/2 + it)")
print(" - Use N = 10^6 as specified")
print(" - Use Kahan compensated summation for numerical stability")
print()
print("3. Coarse scan: t ∈ [80, 90], Δt = 0.1")
print(" - For each t, compute R_comp(t): mean resultant length of composite squarefree phases")
print(" - Identify t* = argmax(R_comp)")
print()
print("4. Fine scan: t ∈ [t* - 0.5, t* + 0.5], Δt = 0.001")
print(" - Compute both R_comp(t) and |D(t)|")
print(" - Identify t_R (peak of R_comp) and t_D (peak of |D|)")
print()
print("5. Report results:")
print(" - t_R: location of R_comp peak")
print(" - t_D: location of |D| peak")
print(" - |t_R - t_D|: distance between peaks")
print(" - Compare to original L_DH zero (t≈85.7) and peak (t≈84.2)")
print()
print("Statistical tests:")
print(" - Rayleigh test for circular uniformity of composite phases")
print(" - Report both p-values and absolute effect sizes (R values)")
print()
print("Expected outcome:")
print(" - R_comp should peak at the off-line zero of L_DH^(ε=0.01)")
print(" - |D(t)| peak may be displaced from this location")
print(" - New peak should be distinct from L_DH locations")
print("="*80)


ANALYSIS PLAN FOR PERTURBED DAVENPORT-HEILBRONN FUNCTION

Objective: Test if R_comp is a general detector for non-multiplicative resonance
by analyzing the perturbed function L_DH^(ε=0.01)

Steps:
1. Implement the perturbed Davenport-Heilbronn function L_DH^(ε=0.01)
 - Use κ' = κ + ε where κ = (√5-1) / (2√(5(√5-1)))
 - Coefficients: a_n = ((1-iκ')/2)χ(n) + ((1+iκ')/2)χ̄(n)
 - Set a_n = 0 for non-squarefree n (critical constraint)

2. Compute partial sum D(t;N) = Σ_{n≤N} a_n / n^(1/2 + it)
 - Use N = 10^6 as specified
 - Use Kahan compensated summation for numerical stability

3. Coarse scan: t ∈ [80, 90], Δt = 0.1
 - For each t, compute R_comp(t): mean resultant length of composite squarefree phases
 - Identify t* = argmax(R_comp)

4. Fine scan: t ∈ [t* - 0.5, t* + 0.5], Δt = 0.001
 - Compute both R_comp(t) and |D(t)|
 - Identify t_R (peak of R_comp) and t_D (peak of |D|)

5. Report results:
 - t_R: location of R_comp peak
 - t_D: location of |D| peak
 - |t_R - t_D|: distance between p

In [2]:

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

print("Step 1: Implementing the perturbed Davenport-Heilbronn function")
print("="*70)

# Define the character χ mod 5 (primitive complex character of order 4)
def chi(n):
 """Character χ modulo 5: χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1, χ(0)=0"""
 n_mod = n % 5
 if n_mod == 0:
 return 0
 elif n_mod == 1:
 return 1
 elif n_mod == 2:
 return 1j
 elif n_mod == 3:
 return -1j
 elif n_mod == 4:
 return -1

def chi_bar(n):
 """Conjugate character χ̄"""
 return np.conj(chi(n))

# Möbius function to identify squarefree numbers
def mobius(n):
 """
 Returns μ(n):
 0 if n is not squarefree
 1 if n is squarefree with even number of prime factors
 -1 if n is squarefree with odd number of prime factors
 """
 if n == 1:
 return 1
 
 # Factor n by trial division
 factors = []
 d = 2
 temp = n
 while d * d <= temp:
 if temp % d == 0:
 count = 0
 while temp % d == 0:
 temp //= d
 count += 1
 if count > 1: # Not squarefree
 return 0
 factors.append(d)
 d += 1
 if temp > 1:
 factors.append(temp)
 
 return (-1) ** len(factors)

# Original DH constant
kappa_original = (np.sqrt(5) - 1) / (2 * np.sqrt(5 * (np.sqrt(5) - 1)))
print(f"Original κ = {kappa_original:.10f}")

# Perturbed constant with ε = 0.01
epsilon = 0.01
kappa_perturbed = kappa_original + epsilon
print(f"Perturbed κ' = κ + {epsilon} = {kappa_perturbed:.10f}")
print()

# Generate coefficients for L_DH^(ε=0.01)
N = 10**6
print(f"Generating coefficients for n ≤ {N:,}...")

# Precompute coefficients
coeffs = np.zeros(N+1, dtype=complex)
for n in range(1, N+1):
 mu_n = mobius(n)
 if mu_n == 0: # Non-squarefree: set coefficient to zero
 coeffs[n] = 0
 else:
 chi_n = chi(n)
 chi_bar_n = chi_bar(n)
 coeffs[n] = ((1 - 1j*kappa_perturbed)/2) * chi_n + ((1 + 1j*kappa_perturbed)/2) * chi_bar_n

print(f"✓ Coefficients generated")
print(f" Non-zero coefficients: {np.count_nonzero(coeffs):,}")
print(f" Zero coefficients (non-squarefree): {N - np.count_nonzero(coeffs):,}")
print()


Step 1: Implementing the perturbed Davenport-Heilbronn function
Original κ = 0.2486028939
Perturbed κ' = κ + 0.01 = 0.2586028939

Generating coefficients for n ≤ 1,000,000...


✓ Coefficients generated
 Non-zero coefficients: 506,604
 Zero coefficients (non-squarefree): 493,396



In [3]:

print("Step 2: Implementing the partial sum D(t;N) with Kahan summation")
print("="*70)

def compute_D_kahan(t, coeffs, N):
 """
 Compute D(t;N) = Σ_{n≤N} a_n / n^(1/2 + it) using Kahan compensated summation
 
 Parameters:
 -----------
 t : float
 Height parameter
 coeffs : array
 Precomputed coefficients a_n
 N : int
 Truncation length
 
 Returns:
 --------
 complex : The partial sum D(t;N)
 """
 # Kahan summation for real and imaginary parts separately
 sum_real = 0.0
 sum_imag = 0.0
 c_real = 0.0 # Compensation for lost low-order bits (real)
 c_imag = 0.0 # Compensation for lost low-order bits (imag)
 
 for n in range(1, N+1):
 if coeffs[n] == 0:
 continue
 
 # Compute term: a_n / n^(1/2 + it)
 # n^(1/2 + it) = n^(1/2) * n^(it) = sqrt(n) * exp(it*log(n))
 sqrt_n = np.sqrt(n)
 phase = t * np.log(n)
 
 # a_n * exp(-i*phase) / sqrt(n)
 cos_phase = np.cos(phase)
 sin_phase = np.sin(phase)
 
 a_real = coeffs[n].real
 a_imag = coeffs[n].imag
 
 # Complex multiplication: (a_real + i*a_imag) * (cos - i*sin) / sqrt_n
 term_real = (a_real * cos_phase + a_imag * sin_phase) / sqrt_n
 term_imag = (a_imag * cos_phase - a_real * sin_phase) / sqrt_n
 
 # Kahan summation - real part
 y_real = term_real - c_real
 t_real = sum_real + y_real
 c_real = (t_real - sum_real) - y_real
 sum_real = t_real
 
 # Kahan summation - imaginary part
 y_imag = term_imag - c_imag
 t_imag = sum_imag + y_imag
 c_imag = (t_imag - sum_imag) - y_imag
 sum_imag = t_imag
 
 return complex(sum_real, sum_imag)

# Test the function at a known point
t_test = 85.0
D_test = compute_D_kahan(t_test, coeffs, N)
print(f"Test computation at t = {t_test}:")
print(f" D(t;N) = {D_test:.6f}")
print(f" |D(t;N)| = {np.abs(D_test):.6f}")
print("✓ Partial sum computation working")
print()


Step 2: Implementing the partial sum D(t;N) with Kahan summation


Test computation at t = 85.0:
 D(t;N) = 1.093266-1.456350j
 |D(t;N)| = 1.821040
✓ Partial sum computation working



In [4]:

print("Step 3: Implementing R_comp metric (composite squarefree phase coherence)")
print("="*70)

def is_prime(n):
 """Simple primality test"""
 if n < 2:
 return False
 if n == 2:
 return True
 if n % 2 == 0:
 return False
 for i in range(3, int(np.sqrt(n)) + 1, 2):
 if n % i == 0:
 return False
 return True

def compute_R_comp(t, coeffs, N):
 """
 Compute R_comp(t): mean resultant length of phases of composite squarefree terms
 
 This metric measures phase coherence among composite squarefree numbers.
 For non-multiplicative functions like L_DH, this peaks at off-line zeros.
 
 Parameters:
 -----------
 t : float
 Height parameter
 coeffs : array
 Precomputed coefficients a_n
 N : int
 Truncation length
 
 Returns:
 --------
 float : Mean resultant length R ∈ [0, 1]
 """
 # Collect phases of composite squarefree terms
 phases = []
 
 for n in range(4, N+1): # Start at 4 (first composite)
 # Check if n is squarefree and composite
 mu_n = mobius(n)
 if mu_n == 0: # Not squarefree
 continue
 if is_prime(n): # Not composite
 continue
 
 # n is composite and squarefree
 if coeffs[n] == 0: # Should not happen for squarefree n
 continue
 
 # Phase of the term a_n / n^(1/2 + it)
 # = arg(a_n) - t*log(n)
 phase = np.angle(coeffs[n]) - t * np.log(n)
 phases.append(phase)
 
 if len(phases) == 0:
 return 0.0
 
 # Convert to unit vectors and compute mean resultant
 phases = np.array(phases)
 exp_phases = np.exp(1j * phases)
 mean_vector = np.mean(exp_phases)
 R = np.abs(mean_vector)
 
 return R

# Test R_comp at a few points
t_values_test = [80.0, 85.0, 90.0]
print("Test R_comp computation:")
for t in t_values_test:
 R = compute_R_comp(t, coeffs, N)
 print(f" t = {t:5.1f}: R_comp = {R:.6f}")
print("✓ R_comp computation working")
print()


Step 3: Implementing R_comp metric (composite squarefree phase coherence)
Test R_comp computation:


 t = 80.0: R_comp = 0.000637


 t = 85.0: R_comp = 0.000778


 t = 90.0: R_comp = 0.000178
✓ R_comp computation working



In [5]:

print("Step 4: COARSE SCAN - t ∈ [80, 90], Δt = 0.1")
print("="*70)

# Coarse scan parameters
t_min_coarse = 80.0
t_max_coarse = 90.0
dt_coarse = 0.1

t_grid_coarse = np.arange(t_min_coarse, t_max_coarse + dt_coarse/2, dt_coarse)
print(f"Scanning {len(t_grid_coarse)} points from t={t_min_coarse} to t={t_max_coarse}")
print("This will take several minutes...")
print()

# Compute R_comp for each t in the coarse grid
R_comp_coarse = []
for i, t in enumerate(t_grid_coarse):
 R = compute_R_comp(t, coeffs, N)
 R_comp_coarse.append(R)
 
 if (i + 1) % 20 == 0:
 print(f" Progress: {i+1}/{len(t_grid_coarse)} ({100*(i+1)/len(t_grid_coarse):.1f}%)")

R_comp_coarse = np.array(R_comp_coarse)

# Find the maximum
idx_max = np.argmax(R_comp_coarse)
t_star = t_grid_coarse[idx_max]
R_max = R_comp_coarse[idx_max]

print()
print(f"✓ Coarse scan complete")
print(f" Maximum R_comp = {R_max:.8f} at t* = {t_star:.1f}")
print()

# Show top 5 peaks
top_indices = np.argsort(R_comp_coarse)[-5:][::-1]
print("Top 5 R_comp values in coarse scan:")
for rank, idx in enumerate(top_indices, 1):
 print(f" {rank}. t = {t_grid_coarse[idx]:.1f}, R_comp = {R_comp_coarse[idx]:.8f}")
print()


Step 4: COARSE SCAN - t ∈ [80, 90], Δt = 0.1
Scanning 101 points from t=80.0 to t=90.0
This will take several minutes...



 Progress: 20/101 (19.8%)


 Progress: 40/101 (39.6%)


 Progress: 60/101 (59.4%)


 Progress: 80/101 (79.2%)


 Progress: 100/101 (99.0%)



✓ Coarse scan complete
 Maximum R_comp = 0.00084317 at t* = 80.5

Top 5 R_comp values in coarse scan:
 1. t = 80.5, R_comp = 0.00084317
 2. t = 80.6, R_comp = 0.00084028
 3. t = 80.4, R_comp = 0.00082138
 4. t = 80.7, R_comp = 0.00080821
 5. t = 80.3, R_comp = 0.00078186



In [6]:

# The computation is still too slow. Let's significantly optimize:
# 1. Precompute squarefree composite indices
# 2. Vectorize the phase computation
# 3. Use smaller fine scan range

print("Optimizing computation by precomputing squarefree composite indices...")

# Precompute indices of squarefree composite numbers
squarefree_composite_indices = []
for n in range(4, N+1):
 mu_n = mobius(n)
 if mu_n == 0: # Not squarefree
 continue
 if is_prime(n): # Not composite
 continue
 squarefree_composite_indices.append(n)

squarefree_composite_indices = np.array(squarefree_composite_indices)
print(f"Found {len(squarefree_composite_indices):,} squarefree composite numbers ≤ {N:,}")

# Precompute coefficient phases and log values for these indices
sc_coeffs = coeffs[squarefree_composite_indices]
sc_coeff_phases = np.angle(sc_coeffs)
sc_log_n = np.log(squarefree_composite_indices)

print("✓ Precomputation complete")
print()

def compute_R_comp_fast(t, sc_coeff_phases, sc_log_n):
 """Fast vectorized R_comp computation"""
 phases = sc_coeff_phases - t * sc_log_n
 exp_phases = np.exp(1j * phases)
 mean_vector = np.mean(exp_phases)
 return np.abs(mean_vector)

# Test the fast version
t_test = 80.5
R_fast = compute_R_comp_fast(t_test, sc_coeff_phases, sc_log_n)
R_slow = compute_R_comp(t_test, coeffs, N)
print(f"Verification at t={t_test}:")
print(f" R_comp (slow): {R_slow:.8f}")
print(f" R_comp (fast): {R_fast:.8f}")
print(f" Difference: {np.abs(R_fast - R_slow):.2e}")
print("✓ Fast version verified")
print()


TimeoutError: Code execution timed out after 1093 seconds